# LoRA fine-tune best VLM (optional)

This notebook fine-tunes **one** VLM using LoRA on Kvasir-VQA x1.
It trains **two variants** to match the paper’s robustness setting:
- **original images**
- **transformed images**

Outputs:
- `results/<model>_lora_original/predictions.jsonl`
- `results/<model>_lora_transformed/predictions.jsonl`
- `metrics.json` for each run

Heavy artifacts (adapters + processor) are saved under `out/`.

Notes:
- Requires GPU + `accelerate` + `peft`.
- Adjust batch size and max length for your GPU memory.


In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

from peft import LoraConfig, get_peft_model


/workspace/vqa-rag/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# Resolve dataset root

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, compute_metrics

MANIFEST = ROOT / "0_dataset_prep" / "out" / "manifest_x1.parquet"
RESULTS_BASE = ROOT / "2_modeling" / "11_lora_finetune" / "results"
OUT_BASE = ROOT / "2_modeling" / "11_lora_finetune" / "out"

RESULTS_BASE.mkdir(parents=True, exist_ok=True)
OUT_BASE.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RESULTS_BASE:", RESULTS_BASE)
print("OUT_BASE:", OUT_BASE)


ROOT: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
RESULTS_BASE: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/11_lora_finetune/results
OUT_BASE: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/11_lora_finetune/out


In [4]:
# Config
MODEL_ID = os.getenv("LORA_MODEL_ID", "google/medgemma-4b-it")
MODEL_NAME = os.getenv("LORA_MODEL_NAME", "medgemma")  # used in output folder names

TRAIN_SPLIT = "train"
EVAL_SPLIT = "test"

RUN_VARIANTS = ["original", "transformed"]  # set to ["original"] if you only want one run

BATCH_SIZE = int(os.getenv("LORA_BATCH_SIZE", "1"))
GRAD_ACCUM_STEPS = int(os.getenv("LORA_GRAD_ACCUM", "4"))
EPOCHS = int(os.getenv("LORA_EPOCHS", "1"))
LR = float(os.getenv("LORA_LR", "2e-4"))
MAX_LENGTH = int(os.getenv("LORA_MAX_LENGTH", "256"))

USE_4BIT = os.getenv("USE_4BIT", "1") == "1"
USE_8BIT = os.getenv("USE_8BIT", "0") == "1"

GEN_KWARGS = {
    "max_new_tokens": 64,
    "do_sample": False,
}


In [5]:
# Load manifest
manifest = pd.read_parquet(MANIFEST)
print("rows:", len(manifest))
print("splits:", manifest["split"].value_counts().to_dict())


rows: 159549
splits: {'train': 143594, 'test': 15955}


In [6]:
# Helpers
STRIP_CHARS = "'""


def format_prompt(question: str, processor) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=True)
    return f"User: <image>
Question: {q}
Assistant:"


def format_prompt_with_answer(question: str, answer: str, processor) -> str:
    q = question.strip()
    a = str(answer).strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
            {"role": "assistant", "content": a},
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=False)
    return f"User: <image>
Question: {q}
Assistant: {a}"


def postprocess(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    for prefix in ["assistant:", "assistant", "answer:"]:
        if t.lower().startswith(prefix):
            t = t[len(prefix):].strip()
    return t


def resolve_image_path(p: str) -> str:
    if p is None:
        return None
    s = str(p)
    path = Path(s)
    if path.exists():
        return str(path)
    if "/Prototyping_reformat/" in s:
        suffix = s.split("/Prototyping_reformat/", 1)[1]
        cand = ROOT.parent / "Prototyping_reformat" / suffix
        if cand.exists():
            return str(cand)
    cand = ROOT / s
    if cand.exists():
        return str(cand)
    cand = ROOT / "0_dataset_prep" / "out" / "images" / "all" / Path(s).name
    if cand.exists():
        return str(cand)
    return str(path)


SyntaxError: unterminated string literal (detected at line 2) (2333202484.py, line 2)

In [ ]:
def load_model(model_id: str):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available for LoRA training")

    if (USE_4BIT or USE_8BIT) and BitsAndBytesConfig is None:
        print("Warning: bitsandbytes not installed; disabling 4/8-bit quantization.")
        use_4bit = False
        use_8bit = False
    else:
        use_4bit = USE_4BIT
        use_8bit = USE_8BIT

    quant_config = None
    if torch.cuda.is_available() and BitsAndBytesConfig is not None:
        if use_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
            )
        elif use_8bit:
            quant_config = BitsAndBytesConfig(load_in_8bit=True)

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

    device_map = {"": 0}
    dtype = torch.float16

    is_vision = getattr(config, "vision_config", None) is not None
    if getattr(config, "model_type", "") in {"qwen2_5_vl", "qwen2_vl", "llava", "idefics2", "idefics3", "fuyu", "blip_2", "git"}:
        is_vision = True

    if is_vision:
        if AutoModelForVision2Seq is None:
            raise RuntimeError("AutoModelForVision2Seq not available. Upgrade transformers.")
        model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            device_map=device_map,
            torch_dtype=dtype,
            quantization_config=quant_config,
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map=device_map,
            torch_dtype=dtype,
            quantization_config=quant_config,
            trust_remote_code=True,
        )

    model.eval()
    return processor, model


In [ ]:
class VQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, max_length: int = 256):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_image_path(row["image_abs_path"] if "image_abs_path" in row else row["image_path"])
        image = Image.open(img_path).convert("RGB")
        prompt = format_prompt_with_answer(row["question"], row["answer"], self.processor)

        inputs = self.processor(
            images=image,
            text=prompt,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.squeeze(0)

        labels = input_ids.clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        item = {
            "input_ids": input_ids,
            "labels": labels,
        }
        if attention_mask is not None:
            item["attention_mask"] = attention_mask
        if "pixel_values" in inputs:
            item["pixel_values"] = inputs["pixel_values"].squeeze(0)
        return item


In [ ]:
def collate_fn(batch):
    keys = batch[0].keys()
    out = {}
    for k in keys:
        out[k] = torch.stack([b[k] for b in batch])
    return out


In [ ]:
def train_lora(df_train: pd.DataFrame, df_eval: pd.DataFrame, variant_name: str):
    results_dir = RESULTS_BASE / f"{MODEL_NAME}_lora_{variant_name}"
    results_dir.mkdir(parents=True, exist_ok=True)

    out_dir = OUT_BASE / f"{MODEL_NAME}_lora_{variant_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    processor, model = load_model(MODEL_ID)

    # LoRA config
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.train()

    train_ds = VQADataset(df_train, processor, max_length=MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    optimizer = AdamW(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"train {variant_name} e{epoch+1}")):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss / GRAD_ACCUM_STEPS
            loss.backward()
            total_loss += loss.item()

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f"epoch {epoch+1} avg loss: {avg_loss:.4f}")

    # save adapters (heavy) to out/
    model.save_pretrained(out_dir / "lora_adapters")
    processor.save_pretrained(out_dir / "processor")

    # evaluate + save predictions/metrics to results/
    preds = []
    for start in tqdm(range(0, len(df_eval), BATCH_SIZE), desc=f"eval {variant_name}"):
        batch = df_eval.iloc[start : start + BATCH_SIZE]
        images = [Image.open(resolve_image_path(p)).convert("RGB") for p in batch["image_abs_path"].tolist()]
        prompts = [format_prompt(q, processor) for q in batch["question"].tolist()]
        inputs = processor(images=images, text=prompts, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        decoded = processor.batch_decode(out, skip_special_tokens=True)
        preds.extend([postprocess(t) for t in decoded])

    out_df = df_eval.copy()
    out_df["pred_raw"] = preds
    out_df["pred_norm"] = out_df["pred_raw"].apply(normalize_answer)

    pred_path = results_dir / "predictions.jsonl"
    out_df.to_json(pred_path, orient="records", lines=True)

    metrics = {"overall": compute_metrics(out_df["pred_norm"], out_df["answer"])}
    with open(results_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print("saved results:", results_dir)
    print("saved heavy artifacts:", out_dir)


In [ ]:
# Run variants

variant_flags = {
    "original": False,
    "transformed": True,
}

for variant in RUN_VARIANTS:
    flag = variant_flags.get(variant)
    if flag is None:
        raise ValueError(f"Unknown variant: {variant}")

    train_df = manifest[(manifest["split"] == TRAIN_SPLIT) & (manifest["is_transformed"] == flag)].copy()
    eval_df = manifest[(manifest["split"] == EVAL_SPLIT) & (manifest["is_transformed"] == flag)].copy()

    print(variant, "train:", len(train_df), "eval:", len(eval_df))
    train_lora(train_df, eval_df, variant)
